# 02 — Caching, Partitioning & Broadcast Join

In [ ]:
import os, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('spark-notebook')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
mode = 'Gluten/Velox' if GLUTEN_ENABLED else 'Vanilla'
print(f'Spark {spark.version}  |  Mode: {mode}')
spark.range(3).show()


## Caching & Persist

In [ ]:
from pyspark import StorageLevel
big_df = spark.range(1_000_000).withColumn('val', F.rand()).withColumn('grp', (F.col('id') % 100).cast('string'))

t0 = time.time(); big_df.count(); print(f'Cold: {time.time()-t0:.2f}s')
big_df.persist(StorageLevel.MEMORY_AND_DISK)
t0 = time.time(); big_df.count(); print(f'Warm: {time.time()-t0:.2f}s')
big_df.unpersist()


## Partitioning

In [ ]:
df = spark.range(500_000).withColumn('group', (F.col('id') % 5).cast('string'))
print('Default partitions:', df.rdd.getNumPartitions())

df_rep = df.repartition(10, 'group')
print('After repartition:', df_rep.rdd.getNumPartitions())

df_coal = df.coalesce(2)
print('After coalesce:', df_coal.rdd.getNumPartitions())


## Broadcast Join

In [ ]:
transactions = spark.range(500_000).withColumn('country_code', (F.col('id') % 4).cast('string'))
countries = spark.createDataFrame([('0','US'),('1','UK'),('2','DE'),('3','FR')], ['country_code','country_name'])

result = transactions.join(F.broadcast(countries), 'country_code')
t0 = time.time()
print('Count:', result.count(), f'  time: {time.time()-t0:.2f}s')
result.groupBy('country_name').count().show()
